# Get sequences features from Ensembl / Uniprot

In [ ]:
import polars as pl
import pandas as pd
import os
from biomart import BiomartServer
from Bio.Seq import Seq
import re

In [ ]:
outputs_dir = "../3_outputs"

## Convert sequences

In [ ]:
metadata = pl.DataFrame(pd.read_csv(f"{metadata_outputs}/varchamp_consolidated_alleles_clinvar_gnomad.tsv", sep="\t"))
priority_col = ["symbol", "nt_change", "ensembl_gene_id", "sequence"]
metadata = metadata.select(priority_col).group_by("symbol").first()

sequences = metadata.select("sequence").to_series().to_list()
nt_changes = metadata.select("nt_change").to_series().to_list()

In [ ]:
# Get wt nucleotide sequences and convert to aa sequences
wt_nt_sequences = []
wt_aa_sequences = []
for seq, nt in zip(sequences, nt_changes):
    pattern = r'(\d+)([A-Za-z])>([A-Za-z])'
    match = re.search(pattern, nt)
    nt_index = int(match.group(1)) - 1
    wt_nt = match.group(2)
    var_nt = match.group(3)

    seq_list = list(seq)

    if seq_list[nt_index] == var_nt:
        seq_list[nt_index] = wt_nt
        nt_seq = ''.join(seq_list)
        aa_seq = Seq(nt_seq).translate()

        wt_nt_sequences.append(nt_seq)
        wt_aa_sequences.append(str(aa_seq))
    else:
        wt_nt_sequences.append(None)
        wt_aa_sequences.append(None)

metadata = metadata.drop("sequence").with_columns(
    pl.Series(name="aa_sequence", values=wt_aa_sequences),
    pl.Series(name="wt_sequence", values=wt_nt_sequences),
)

In [ ]:
ensembl_gene_ids = metadata.select('ensembl_gene_id').to_series().unique().to_list()
server = BiomartServer("http://www.ensembl.org/biomart/martservice")
mart = server.datasets['hsapiens_gene_ensembl']

attributes = [
    'ensembl_gene_id',
    'ensembl_peptide_id',
    'peptide'
]


protein_ids = list(metadata["ensembl_gene_id"].unique())
batch_size = 100  
results = []

# Query BioMart in batches
for i in range(0, len(protein_ids), batch_size):
    batch = protein_ids[i : i + batch_size]
    
    filters = {
        'ensembl_gene_id': batch
    }
    
    try:
        response = mart.search({"attributes": attributes, "filters": filters})
        data = [line.decode("utf-8").split("\t") for line in response.iter_lines()]
        df_batch = pl.DataFrame(pd.DataFrame(data))
        df_batch.columns = ["amino_acid_sequence", "ensembl_gene_id", "ensembl_peptide_id"]
        results.append(df_batch)
    except Exception as e:
        print(f"Batch {i//batch_size + 1} failed: {e}")

all_aa_seqs = pl.concat(results)

In [ ]:
# Need to find the perfect matches, then merge with our data

## Get sequence features

In [ ]:
# process metadata
metadata = pl.DataFrame(pd.read_csv(f"{outputs_dir}/varchamp_consolidated_alleles_clinvar_gnomad.tsv", sep="\t"))
mut_cols = ["mutation_id_rc4", "mutation_id_cegs2", "mutation_id_edgotyping3"]
id_vars = [col for col in metadata.columns if col not in mut_cols]
metadata = metadata.unpivot(index=id_vars, 
                            on=mut_cols,
                            variable_name="mutation_collection",
                            value_name="mut_id")

priority_col = ["orf_id", "mut_id", "symbol", "aa_change", "ensembl_gene_id", "clinvar_clnsig_clean", "gnomad_af", "StarStatus"]
metadata = metadata.select(priority_col).filter(~pl.col("mut_id").is_null()).with_columns(
    pl.col("mut_id").cast(pl.Int64).alias("mut_id"),
    pl.col("orf_id").cast(pl.Int64).alias("orf_id"),
    pl.concat_str([pl.col("symbol"), pl.col("aa_change")], separator="_").alias("allele_0")
)

In [ ]:
# Get sequence features
server = BiomartServer("http://www.ensembl.org/biomart/martservice")
mart = server.datasets['hsapiens_gene_ensembl']

attributes = [
    "ensembl_peptide_id_version", "external_gene_name", "signalp", 
    "signalp_start", "signalp_end", "tmhmm", "tmhmm_start", 
    "tmhmm_end", "ncoils", "ncoils_start", "ncoils_end", "seg", 
    "seg_start", "seg_end", "pfam", "pfam_start", "pfam_end"
]

protein_ids = list(metadata["ensembl_gene_id"].unique())
batch_size = 100  
results = []

# Query BioMart in batches
for i in range(0, len(protein_ids), batch_size):
    batch = protein_ids[i : i + batch_size]
    
    filters = {"ensembl_gene_id": batch}
    
    try:
        response = mart.search({"attributes": attributes, "filters": filters})
        data = [line.decode("utf-8").split("\t") for line in response.iter_lines()]
        df_batch = pl.DataFrame(pd.DataFrame(data))
        results.append(df_batch)
        print(f"Batch {i//batch_size + 1} success!")
    except Exception as e:
        print(f"Batch {i//batch_size + 1} failed: {e}")
        

seq_features = pl.concat(results)

Batch 1 success!
Batch 2 success!
Batch 3 success!
Batch 4 success!
Batch 5 success!
Batch 6 success!
Batch 7 success!
Batch 8 success!
Batch 9 success!
Batch 10 success!
Batch 11 success!
Batch 12 success!
Batch 13 success!
Batch 14 success!
Batch 15 success!
Batch 16 success!
Batch 17 success!


- There are multiple peptides for each Ensembl gene
- The coordinates for the domains (helix, unstructured region, etc) are at the Ensembl protein level (ie. describe amino acids)
- We have sequences in terms of nucleotides that are mapped to the closest Ensembl gene
- We need to get the Ensembl protein that exactly matches our sequence because otherwise the domain coordinates may not match
- Since a gene has multiple associated proteins, how do we choose the right one? 
- Convert our nucleotide sequences into amino acid sequences and search for exact matches among all peptide sequences for the list of Ensembl genes